In [1]:
import pandas as pd

dfs = []
for i in [1, 2, 3]:
    df = pd.read_excel(f"Results_19/ai_auto_eval_19_deepseek-chat-{i}.xlsx")
    # Rename columns to avoid collision
    df = df.rename(columns={
        f"deepseek-chat-{i}_Answer": f"Answer_{i}",
        f"deepseek-chat-{i}_Evidence": f"Evidence_{i}",
        f"deepseek-chat-{i}_Rationale": f"Rationale_{i}",
    })
    dfs.append(df[["PMID", "QID", "Question", "Human_Answer", f"Answer_{i}", f"Evidence_{i}", f"Rationale_{i}"]])

# Merge step by step
merged = dfs[0]
for df in dfs[1:]:
    merged = pd.merge(merged, df, on=["PMID", "QID", "Question", "Human_Answer"], how="outer")

merged.to_excel("Results_19/merged_3runs.xlsx", index=False)


In [4]:
ADJUDICATOR_PROMPT = """
You are an Expert Adjudicator Agent in a virtual scientific lab. 
Your task is to synthesize the most accurate, evidence-grounded answer from three AI systems that attempted to extract information from a research paper. 
Each AI may have made mistakes. You must reason critically, check for validity, and produce a single best final answer.

---
**Question:**
{question}

**Human gold-standard answer (for reference only; do NOT copy):**
{human_answer}

**Three AI answers:**
1️⃣ Answer 1: {answer1}
   Evidence: {evidence1}
   Rationale: {rationale1}

2️⃣ Answer 2: {answer2}
   Evidence: {evidence2}
   Rationale: {rationale2}

3️⃣ Answer 3: {answer3}
   Evidence: {evidence3}
   Rationale: {rationale3}

---
### Your Role:
You are both an **Adjudicator** and a **Critic Agent**. Follow these reasoning steps explicitly before deciding:

1. **Check for blank or incomplete answers.**
   - If any answer is missing or blank, attempt to infer the likely answer based on the paper context and other answers.

2. **Evidence validity audit.**
   - Ask yourself: does the cited evidence describe results from *this study*, or is it describing prior work referenced in the paper?
   - Reject or down-weight evidence that refers to other studies.

3. **Domain knowledge check.**
   - Make sure the answer uses correct domain understanding.
   - Use your knowledge in biomedical research (HIV for this case) to interpret terms correctly.

4. **Cross-verification.**
   - If the three AI answers agree but are likely wrong in the same way, question the common assumption.
   - If they disagree, analyze which one is more consistent with scientific logic and the context of the question.

5. **Missing evidence.**
   - If an answer has no evidence or rationale, note it unless the conclusion is clearly correct from context.

6. **external verification (conceptual).**
   - For claims that can be checked from general scientific knowledge (e.g., terminology, definitions), verify them mentally.
   - If the answer contains external sources, such as a link or certain serial numbers that can be looked up on other websites, always look them up to verify.

7. **Resolve and produce final synthesis.**
   - Construct the best possible final answer in your own words.
   - Be concise but specific.
   - Provide a short rationale for your adjudication.

---

### Output format:
Respond in **JSON only**, using this schema:

{{
  "Final_Answer": "your single best adjudicated answer",
  "Rationale": "your reasoning and evaluation of the three answers, including which one(s) were partly correct or wrong and why",
  "Confidence": 0.0-1.0
}}
"""



In [10]:
import re
import json

def safe_parse_json(text):
    """Try to extract and parse JSON object from text output."""
    try:
        # Extract JSON block if there’s extra text
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            json_str = match.group(0)
            return json.loads(json_str)
    except Exception:
        pass
    return None

In [11]:
from pathlib import Path
import pandas as pd

# ===========================
# 🔧 CONFIGURATION
# ===========================
PROVIDER = "deepseek"      # options: "deepseek", "openai", "claude"
MODEL = "deepseek-chat"    # change to "gpt-4o-mini", "claude-3-opus", etc.
OUTPUT_FILE = f"Results_19/ai_adjudicated_{MODEL}.xlsx"

# Insert your API keys here
openai_api_key = ""
claude_api_key = ""
deepseek_api_key = ""
# ===========================
# 🔌 CLIENT SETUP
# ===========================
if PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=openai_api_key)

elif PROVIDER == "deepseek":
    from openai import OpenAI
    client = OpenAI(
        api_key=deepseek_api_key,
        base_url="https://api.deepseek.com"
    )

elif PROVIDER == "claude":
    from anthropic import Anthropic
    client = Anthropic(api_key=claude_api_key)

else:
    raise ValueError("Unsupported provider. Choose from: openai, deepseek, claude.")

# ===========================
# 🧠 HELPER FUNCTION
# ===========================
def safe_parse_json(text):
    """Try to extract and parse JSON object from messy model output."""
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            json_str = match.group(0)
            return json.loads(json_str)
    except Exception:
        pass
    return None

# ===========================
# 🧩 LOAD MERGED INPUT FILE
# ===========================
merged = pd.read_excel("Results_19/merged_3runs.xlsx")

# ===========================
# 🚀 RUN ADJUDICATION
# ===========================
results = []
for _, row in merged.iterrows():
    prompt = ADJUDICATOR_PROMPT.format(
        question=row["Question"],
        human_answer=row["Human_Answer"],
        answer1=row["Answer_1"],
        answer2=row["Answer_2"],
        answer3=row["Answer_3"],
        evidence1=row.get("Evidence_1", ""),
        evidence2=row.get("Evidence_2", ""),
        evidence3=row.get("Evidence_3", ""),
        rationale1=row.get("Rationale_1", ""),
        rationale2=row.get("Rationale_2", ""),
        rationale3=row.get("Rationale_3", "")
    )

    try:
        # ===== Call the right API based on provider =====
        if PROVIDER == "openai":
            response = client.responses.create(
                model=MODEL,
                input=prompt,
                temperature=0
            )
            text = response.output[0].content[0].text

        elif PROVIDER == "deepseek":
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            text = response.choices[0].message.content

        elif PROVIDER == "claude":
            response = client.messages.create(
                model=MODEL,
                max_tokens=2048,
                temperature=0,
                messages=[{"role": "user", "content": prompt}]
            )
            text = response.content[0].text

        # ===== Try to parse JSON, fall back if not valid =====
        parsed = safe_parse_json(text)
        if parsed:
            final_answer = parsed.get("Final_Answer", "")
            rationale = parsed.get("Rationale", "")
            confidence = parsed.get("Confidence", "")
        else:
            print(f"⚠️ JSON parse error for PMID {row['PMID']} QID {row['QID']}")
            final_answer = text.strip()
            rationale = ""
            confidence = ""

        results.append({
            "PMID": row["PMID"],
            "QID": row["QID"],
            "Final_Answer": final_answer,
            "Rationale": rationale,
            "Confidence": confidence,
            "Raw_Text": text
        })

    except Exception as e:
        print(f"❌ Error on PMID {row['PMID']}, QID {row['QID']}: {e}")
        results.append({
            "PMID": row["PMID"],
            "QID": row["QID"],
            "Final_Answer": "",
            "Rationale": "",
            "Confidence": "",
            "Raw_Text": f"Error: {e}"
        })

# ===========================
# 💾 SAVE RESULTS
# ===========================
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(results).to_excel(OUTPUT_FILE, index=False)
print(f"✅ Adjudication complete. Results saved to {OUTPUT_FILE}")

✅ Adjudication complete. Results saved to Results_19/ai_adjudicated_deepseek-chat.xlsx
